# Structured Outputs
This notebook demonstrates how to work with different output formats and parsers when working with Language Models (LLM).

## What we'll learn:
- Basic string output parsing
- Working with tools and structured outputs
- Using Pydantic models for type validation
- Understanding different parser types and their use cases

### Setup

In [1]:
import os
import json
import chromadb
from pydantic import BaseModel, Field
from typing import List,Any, Annotated
from chromadb.utils import embedding_functions
from lib.parsers import (
    PydanticOutputParser    
)
from dotenv import load_dotenv

In [2]:
load_dotenv()

True

## ChromaDB with Default Embedding Function

In [3]:
chroma_client = chromadb.PersistentClient(path="./chroma_db")

In [4]:
collection = chroma_client.create_collection(
    name="demo"
)

In [5]:
data_dir = "games"

for file_name in sorted(os.listdir(data_dir)):
    if not file_name.endswith(".json"):
        continue

    file_path = os.path.join(data_dir, file_name)
    with open(file_path, "r", encoding="utf-8") as f:
        game = json.load(f)

    # You can change what text you want to index
    content = f"[{game['Platform']}] {game['Name']} ({game['YearOfRelease']}) - {game['Description']}"

    # Use file name (like 001) as ID
    doc_id = os.path.splitext(file_name)[0]

    collection.add(
        ids=[doc_id],
        documents=[content],
        metadatas=[game]
    )

In [6]:
collection.count()

15

In [7]:
collection.peek(1)

{'ids': ['001'],
 'embeddings': array([[-4.76837307e-02, -1.36362854e-02, -2.10048649e-02,
         -2.89396588e-02,  1.70632172e-02, -4.05941196e-02,
          1.59703698e-02,  1.06667746e-02, -4.79368456e-02,
         -4.16721143e-02, -3.25565077e-02, -1.52356578e-02,
          1.93945765e-02, -4.66961674e-02, -3.37734558e-02,
         -1.36071388e-02,  7.48895332e-02,  6.96156174e-02,
          7.18380660e-02, -2.69509666e-02, -1.07769445e-02,
         -7.88036138e-02, -3.30629833e-02,  3.80312614e-02,
         -5.10617271e-02,  2.15464421e-02, -4.50067408e-02,
          2.55763382e-02, -9.91723910e-02, -5.21281250e-02,
         -4.38892245e-02,  5.53388288e-03,  7.51421675e-02,
          2.68152691e-02, -1.96971036e-02, -9.63657498e-02,
         -3.82603146e-02, -1.22499391e-02, -9.61586684e-02,
          4.07790486e-03, -3.92131209e-02, -4.16620821e-02,
          1.49855176e-02,  1.75323039e-02,  1.10037260e-01,
          6.97333086e-03,  5.50322793e-03, -9.23289061e-02,
         

In [8]:
collection.query(
    query_texts=["gadget"],
    n_results=2,
    include=['metadatas', 'documents', 'distances']
)

{'ids': [['003', '008']],
 'embeddings': None,
 'documents': [['[PlayStation 3] Gran Turismo 5 (2010) - A comprehensive racing simulator featuring a vast selection of vehicles and tracks, with realistic driving physics.',
   '[Super Nintendo Entertainment System (SNES)] Super Mario World (1990) - A classic platformer where Mario embarks on a quest to save Princess Toadstool and Dinosaur Land from Bowser.']],
 'uris': None,
 'included': ['metadatas', 'documents', 'distances'],
 'data': None,
 'metadatas': [[{'Platform': 'PlayStation 3',
    'Description': 'A comprehensive racing simulator featuring a vast selection of vehicles and tracks, with realistic driving physics.',
    'YearOfRelease': 2010,
    'Publisher': 'Sony Computer Entertainment',
    'Name': 'Gran Turismo 5',
    'Genre': 'Racing'},
   {'YearOfRelease': 1990,
    'Genre': 'Platformer',
    'Publisher': 'Nintendo',
    'Name': 'Super Mario World',
    'Platform': 'Super Nintendo Entertainment System (SNES)',
    'Descript

In [9]:
print(collection._embedding_function.name())

default


In [10]:
size = len(collection.peek(1)['embeddings'][0])
print(f"Size of the embeddings array: {size}")

Size of the embeddings array: 384


## OpenAI Embeddings

In [11]:
chroma_client.delete_collection(name="demo")

In [12]:
embeddings_fn = embedding_functions.OpenAIEmbeddingFunction(
    api_key=os.getenv("OPENAI_API_KEY")
)

In [13]:
collection = chroma_client.create_collection(
    name="demo",
    embedding_function=embeddings_fn
)

In [14]:
for documentid in doc_id:
    collection.add(
        documents=content,
        ids=documentid
    )

In [15]:
collection.query(
    query_texts=["gadget"],
    n_results=2,
    include=['metadatas', 'documents', 'distances']
)

{'ids': [['1', '5']],
 'embeddings': None,
 'documents': [["[Xbox Series X|S] Halo Infinite (2021) - The latest installment in the Halo franchise, featuring Master Chief's return in a new open-world setting.",
   "[Xbox Series X|S] Halo Infinite (2021) - The latest installment in the Halo franchise, featuring Master Chief's return in a new open-world setting."]],
 'uris': None,
 'included': ['metadatas', 'documents', 'distances'],
 'data': None,
 'metadatas': [[None, None]],
 'distances': [[0.28082704544067383, 0.28082704544067383]]}

In [16]:
print(collection._embedding_function.name())

openai


In [ ]:
size = len(collection.peek(1)['embeddings'][0])
print(f"Size of the embeddings array: {size}")

Size of the embeddings array: 1536
